# Adaptive RAG 测试 Notebook

本 notebook 用于测试 Adaptive_RAG.py 的自适应策略。

## 核心特性
- **自适应策略选择**：根据问题类型自动选择最优 RAG 策略
  - 事实题 → Baseline RAG（直接检索）
  - 复杂题/推理题 → Advanced RAG（问题分解 + Rerank）
- **统一文档处理**：chunk_size=600，与 baseline 一致
- **完整评测支持**：集成 Ragas 5 项指标

In [10]:
# 导入 Adaptive RAG
from Adaptive_RAG import (
    AdaptiveRAG,
    run_ragas_evaluation,
    sample_stratified_evaluation
)

# 导入评测集
from evaluation_dataset import eval_questions_50, eval_ground_truths_50

print("✅ 导入成功")

✅ 导入成功


## 1. 初始化 Adaptive RAG 系统

In [11]:
# PDF URL
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"

# 创建系统
rag = AdaptiveRAG(
    pdf_url=pdf_url,
    chunk_size=600,    # 与 baseline 一致
    chunk_overlap=60
)

🚀 初始化 Adaptive RAG 系统
正在加载 PDF: https://arxiv.org/pdf/1706.03762.pdf
✅ 加载完成，共 15 页
✅ 清洗完成，处理 15 页
✅ 切分完成，共 78 个片段
✅ 混合检索器创建完成
✅ Adaptive RAG 系统初始化完成



## 2. 测试不同类型的问题

In [12]:
# 测试问题
test_questions = [
    # 事实题（应使用 Baseline）
    "Transformer 的 base 配置使用了多少层？",
    "d_model 在 base 配置中取值是多少？",
    
    # 复杂题（应使用 Advanced）
    "Transformer 为什么需要多头注意力机制？",
    "残差连接和层归一化的作用是什么？",
    
    # 推理题（应使用 Advanced）
    "如果去掉残差连接，模型训练会遇到什么问题？",
    "为什么自注意力机制相比循环网络能更好地捕捉长距离依赖？"
]

print("="*80)
print("🧪 测试自适应策略选择")
print("="*80 + "\n")

for question in test_questions:
    answer, strategy, contexts = rag.answer(question)
    print(f"\n{'='*80}")
    print(f"问题: {question}")
    print(f"策略: {strategy}")
    print(f"检索上下文数: {len(contexts)}")
    print(f"答案: {answer}")
    print(f"{'='*80}")

🧪 测试自适应策略选择

📝 [Baseline] 事实题: Transformer 的 base 配置使用了多少层？...

问题: Transformer 的 base 配置使用了多少层？
策略: baseline
检索上下文数: 3
答案: Transformer 的 base 配置使用了 6 层。
📝 [Baseline] 事实题: d_model 在 base 配置中取值是多少？...

问题: d_model 在 base 配置中取值是多少？
策略: baseline
检索上下文数: 3
答案: d_model 在 base 配置中取值是 512。
🔍 [Advanced] complex题: Transformer 为什么需要多头注意力机制？...


KeyboardInterrupt: 

## 3. 查看策略使用统计

In [ ]:
# 获取统计信息
stats = rag.get_stats()
total = sum(stats.values())

print("="*80)
print("📊 策略使用统计")
print("="*80)
print(f"事实题 (Baseline): {stats['simple']} ({stats['simple']/total*100:.1f}%)")
print(f"复杂题 (Advanced): {stats['complex']} ({stats['complex']/total*100:.1f}%)")
print(f"推理题 (Advanced): {stats['reasoning']} ({stats['reasoning']/total*100:.1f}%)")
print(f"总计: {total} 题")
print("="*80)

📊 策略使用统计
事实题 (Baseline): 3 (50.0%)
复杂题 (Advanced): 2 (33.3%)
推理题 (Advanced): 1 (16.7%)
总计: 6 题


## 4. 小样本评测（分层采样）

In [13]:
# 分层采样
eval_indices = sample_stratified_evaluation(total_samples=10)

# 运行评测
sample_report = run_ragas_evaluation(
    adaptive_rag=rag,
    eval_questions=eval_questions_50,
    eval_ground_truths=eval_ground_truths_50,
    indices=eval_indices
)

✅ 分层采样完成：共 10 题
   - 基础知识题: 8 题
   - 复杂题: 1 题
   - 推理题: 1 题
   - 采样题号: [4, 6, 13, 26, 27, 30, 33, 36, 43, 49]

[1/10] Transformer 在每个子层外部使用了哪两种稳定训练的结构？...
📝 [Baseline] 事实题: Transformer 在每个子层外部使用了哪两种稳定训练的结构？...
[2/10] 为什么注意力分数要除以 sqrt(d_k)？...
🔍 [Advanced] complex题: 为什么注意力分数要除以 sqrt(d_k)？...
[3/10] 编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？...
📝 [Baseline] 事实题: 编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？...
[4/10] big 模型的训练耗时大约是多少？...
📝 [Baseline] 事实题: big 模型的训练耗时大约是多少？...
[5/10] Transformer base 模型大约有多少参数？...
📝 [Baseline] 事实题: Transformer base 模型大约有多少参数？...
[6/10] 论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？...
📝 [Baseline] 事实题: 论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？...
[7/10] 循环层每层的时间复杂度是多少？...
📝 [Baseline] 事实题: 循环层每层的时间复杂度是多少？...
[8/10] 《Attention Is All You Need》主要验证的是哪类任务？...
📝 [Baseline] 事实题: 《Attention Is All You Need》主要验证的是哪类任务？...
[9/10] 受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？...
📝 [Baseline] 事实题: 受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？...
[10/10] 如果在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可能在

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   2%|▏         | 1/50 [00:08<06:36,  8.09s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   8%|▊         | 4/50 [00:15<02:18,  3.02s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:  14%|█▍        | 7/50 [00:17<01:09,  1.61s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token u


📊 策略使用统计
事实题 (Baseline): 8 (80.0%)
复杂题 (Advanced): 1 (10.0%)
推理题 (Advanced): 1 (10.0%)


## 5. 查看评测结果

In [15]:
import pandas as pd

# 显示结果
metric_cols = [
    col for col in [
        "faithfulness",
        "answer_relevancy",
        "context_recall",
        "llm_context_precision_with_reference",
        "answer_correctness"
    ]
    if col in sample_report.columns
]

print("="*80)
print("📊 Adaptive RAG 评测结果")
print("="*80)
print(f"\n{'指标':<30} {'平均分':>10}")
print("-"*50)
print(f"{'Faithfulness (忠实性)':<30} {sample_report['faithfulness'].mean():>10.4f}")
print(f"{'Answer Relevancy (答案相关性)':<30} {sample_report['answer_relevancy'].mean():>10.4f}")
print(f"{'Context Recall (上下文召回)':<30} {sample_report['context_recall'].mean():>10.4f}")
print(f"{'Context Precision (上下文精度)':<30} {sample_report['llm_context_precision_with_reference'].mean():>10.4f}")
print(f"{'Answer Correctness (答案正确性)':<30} {sample_report['answer_correctness'].mean():>10.4f}")
print("-"*50)
print(f"{'总体平均分':<30} {sample_report[metric_cols].mean().mean():>10.4f}")
print("="*80)

# 详细结果
display_cols = ["user_input", "response", "faithfulness", "answer_relevancy", "answer_correctness"]
print("\n详细结果：")
display(sample_report[display_cols].head(10))

📊 Adaptive RAG 评测结果

指标                                    平均分
--------------------------------------------------
Faithfulness (忠实性)                 0.7100
Answer Relevancy (答案相关性)           0.7998
Context Recall (上下文召回)             0.9000
Context Precision (上下文精度)          0.5833
Answer Correctness (答案正确性)         0.5620
--------------------------------------------------
总体平均分                              0.7110

详细结果：


,user_input,response,faithfulness,answer_relevancy,answer_correctness
0,Transformer 在每个子层外部使用了哪两种稳定训练的结构？,Transformer 在每个子层外部使用了残差连接和丢弃（dropout）结构。,0.5,0.775303,0.505672
1,为什么注意力分数要除以 sqrt(d_k)？,注意力分数要除以 sqrt(d_k) 是为了对点积进行缩放，防止其数值过大，从而避免梯度消失...,0.8,0.964088,0.521990
2,编码器-解码器注意力中的 Query、Key、Value 分别来自哪里？,在编码器-解码器注意力中，Query 来自解码器的输出，Key 和 Value 来自编码器的输出。,1.0,0.982752,0.737804
3,big 模型的训练耗时大约是多少？,big 模型的训练耗时大约是 3.5 天。,1.0,0.833948,0.724552
4,Transformer base 模型大约有多少参数？,Transformer base 模型有 65 × 10^6 参数。,1.0,0.833471,0.995971
5,论文报告的 Transformer big 在英法任务上的 BLEU 分数是多少？,根据论文，Transformer big 在英法任务上的 BLEU 分数是 41.0。,1.0,0.937673,0.223933
6,循环层每层的时间复杂度是多少？,循环层每层的时间复杂度是O(n · d²)。,1.0,0.940556,0.997302
7,《Attention Is All You Need》主要验证的是哪类任务？,《Attention Is All You Need》主要验证的是序列到序列的任务。,0.0,0.943627,0.163014
8,受限自注意力在每层的时间复杂度是多少，相比一般自注意力和循环层各有什么优劣？,受限自注意力在每层的时间复杂度是O(r · n · d)。相比一般自注意力的O(n² · d...,0.8,0.786810,0.685804
9,如果在序列很长的推理场景下运行 Transformer，相比循环网络，主要的性能瓶颈可能在哪里？,我不知道。,0.0,0.000000,0.063673


## 6. 与 Baseline/Advanced 对比

In [16]:
# 对比三种方法的评测结果
# （需要先运行 rag_baseline.ipynb 和 advanced_rag.ipynb 获取它们的结果）

comparison = {
    "方法": ["Baseline", "Advanced", "Adaptive"],
    "Faithfulness": [1.0, 0.207, sample_report['faithfulness'].mean()],
    "Answer Relevancy": [0.899, 0.865, sample_report['answer_relevancy'].mean()],
    "Context Recall": [1.0, 0.533, sample_report['context_recall'].mean()],
    "Context Precision": [0.85, 0.167, sample_report['llm_context_precision_with_reference'].mean()],
    "Answer Correctness": [None, None, sample_report['answer_correctness'].mean()]
}

df_comparison = pd.DataFrame(comparison)
print("\n三种方法对比：")
display(df_comparison)


三种方法对比：


,方法,Faithfulness,Answer Relevancy,Context Recall,Context Precision,Answer Correctness
0,Baseline,1.000,0.899000,1.000,0.850000,NaN
1,Advanced,0.207,0.865000,0.533,0.167000,NaN
2,Adaptive,0.710,0.799823,0.900,0.583333,0.561972


## 7. 全量评测（可选）

In [ ]:
# 全量 50 题评测（需要较长时间）
# full_report = run_ragas_evaluation(
#     adaptive_rag=rag,
#     eval_questions=eval_questions_50,
#     eval_ground_truths=eval_ground_truths_50
# )

# print("\n全量评测结果：")
# display(full_report)